<a href="https://colab.research.google.com/github/ChandanaDG03/fintrack-personal-finance/blob/main/Personal_Finance_Tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import logging
import sys
import tempfile
from pathlib import Path
import functools
import time
import uuid
import sqlite3
import csv
import json
import random
from contextlib import contextmanager
from collections import defaultdict
from dataclasses import dataclass, field, asdict
from datetime import date, datetime, timedelta
from enum import Enum
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe backend
import matplotlib.pyplot as plt

# --- Paths -------------------------------------------------------
BASE_DIR = Path(tempfile.gettempdir()) / "fintrack_demo"
BASE_DIR.mkdir(parents=True, exist_ok=True)
DEFAULT_DB_PATH = BASE_DIR / "fintrack.db"

# --- Domain defaults ---------------------------------------------
CURRENCY_SYMBOL = "$"
DATE_FORMAT = "%Y-%m-%d"

# --- Logging Configuration ---------------------------------------
def configure_logging(level: int = logging.INFO) -> logging.Logger:
    """Configure and return a named logger for FinTrack."""
    logger = logging.getLogger("fintrack")
    if logger.handlers:
        return logger  # already configured
    logger.setLevel(level)
    fmt = logging.Formatter("%(levelname)-8s | %(name)s | %(message)s")
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    return logger

log = configure_logging()

# --- Custom Exceptions -------------------------------------------
class FinTrackError(Exception):
    """Base class for all FinTrack-specific errors."""
class ValidationError(FinTrackError):
    """Raised when user-supplied data fails validation."""
class InsufficientDataError(FinTrackError):
    """Raised when a report is requested but no data is available."""
class DatabaseError(FinTrackError):
    """Raised when an underlying database operation fails."""

# --- Decorators for Logging, Retry, Validation -------------------
def log_call(func):
    """Decorator: log entry, exit, and duration of the function call."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        log.debug("CALL %s", func.__name__)
        try:
            result = func(*args, **kwargs)
        except Exception:
            log.exception("ERROR in %s", func.__name__)
            raise
        duration_ms = (time.perf_counter() - start) * 1000
        log.debug("DONE %s in %.2f ms", func.__name__, duration_ms)
        return result
    return wrapper

def retry(times: int = 3, delay: float = 0.2, exceptions: tuple = (Exception,)):
    """Decorator: retry the function up to `times` if specified exceptions occur."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:
                    last_exc = exc
                    log.warning("Attempt %d/%d of %s failed: %s", attempt, times, func.__name__, exc)
                    time.sleep(delay)
            raise last_exc
        return wrapper
    return decorator

def validate_positive(*field_names):
    """Decorator: ensure named keyword arguments are positive numbers."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for name in field_names:
                if name in kwargs:
                    value = kwargs[name]
                    if not isinstance(value, (int, float)) or value <= 0:
                        raise ValidationError(f"'{name}' must be a positive number, got {value!r}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

# --- Data Model Classes ------------------------------------------
class TransactionType(str, Enum):
    INCOME = "income"
    EXPENSE = "expense"

    @classmethod
    def from_string(cls, value: str):
        try:
            return cls(value.strip().lower())
        except ValueError as exc:
            valid = ", ".join(t.value for t in cls)
            raise ValidationError(f"Invalid type {value!r}. Expected: {valid}") from exc

def _new_id() -> str:
    return uuid.uuid4().hex[:12]  # short unique ID

@dataclass(slots=True)
class Transaction:
    amount: float
    category: str
    ttype: TransactionType
    note: str = ""
    txn_date: date = field(default_factory=date.today)
    id: str = field(default_factory=_new_id)

    def __post_init__(self):
        if self.amount <= 0:
            raise ValidationError("Transaction amount must be positive.")
        self.category = self.category.strip().lower()
        if not self.category:
            raise ValidationError("Transaction category cannot be empty.")
        # If ttype given as string, parse it
        if isinstance(self.ttype, str):
            self.ttype = TransactionType.from_string(self.ttype)
        # If txn_date given as string, parse it
        if isinstance(self.txn_date, str):
            self.txn_date = datetime.strptime(self.txn_date, DATE_FORMAT).date()

    @property
    def signed_amount(self) -> float:
        # Use == for enum comparison (safer than 'is')
        return self.amount if self.ttype == TransactionType.INCOME else -self.amount

    def to_row(self):
        return (self.id, self.amount, self.category, self.ttype.value, self.note,
                self.txn_date.strftime(DATE_FORMAT))

    def to_dict(self):
        data = asdict(self)
        data["ttype"] = self.ttype.value
        data["txn_date"] = self.txn_date.strftime(DATE_FORMAT)
        return data

    @classmethod
    def from_row(cls, row):
        txn_id, amount, category, ttype, note, txn_date = row
        return cls(amount=amount, category=category, ttype=TransactionType(ttype),
                   note=note, txn_date=datetime.strptime(txn_date, DATE_FORMAT).date(), id=txn_id)

@dataclass(slots=True)
class Budget:
    category: str
    monthly_limit: float

    def __post_init__(self):
        self.category = self.category.strip().lower()
        if not self.category:
            raise ValidationError("Budget category cannot be empty.")
        if self.monthly_limit <= 0:
            raise ValidationError("Budget limit must be positive.")

# --- Sanity Check for Data Model ----------------------------------
demo = Transaction(amount=1200, category="Salary", ttype="income")
print("Created:", demo.to_dict(), "| signed:", demo.signed_amount)

# --- Database Access Layer ---------------------------------------
_SCHEMA = """
CREATE TABLE IF NOT EXISTS transactions (
    id TEXT PRIMARY KEY,
    amount REAL NOT NULL CHECK (amount > 0),
    category TEXT NOT NULL,
    ttype TEXT NOT NULL CHECK (ttype IN ('income','expense')),
    note TEXT DEFAULT '',
    txn_date TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS budgets (
    category TEXT PRIMARY KEY,
    monthly_limit REAL NOT NULL CHECK (monthly_limit > 0)
);
CREATE INDEX IF NOT EXISTS idx_txn_date ON transactions (txn_date);
"""

class Database:
    def __init__(self, db_path=DEFAULT_DB_PATH) -> None:
        self.db_path = str(db_path)
        self._initialise()

    @contextmanager
    def _connect(self):
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        try:
            yield conn
            conn.commit()
        except sqlite3.Error as exc:
            conn.rollback()
            raise DatabaseError(str(exc)) from exc
        finally:
            conn.close()

    @log_call
    def _initialise(self):
        with self._connect() as conn:
            conn.executescript(_SCHEMA)

    @retry(times=3, delay=0.1, exceptions=(DatabaseError,))
    def add_transaction(self, txn: Transaction) -> str:
        with self._connect() as conn:
            conn.execute(
                "INSERT INTO transactions (id, amount, category, ttype, note, txn_date)"
                " VALUES (?, ?, ?, ?, ?, ?)",
                txn.to_row()
            )
        return txn.id

    def delete_transaction(self, txn_id: str) -> bool:
        with self._connect() as conn:
            return conn.execute("DELETE FROM transactions WHERE id = ?", (txn_id,)).rowcount > 0

    def iter_transactions(self, *, start=None, end=None, category=None, ttype=None):
        clauses, params = [], []
        if start:
            clauses.append("txn_date >= ?"); params.append(start.strftime(DATE_FORMAT))
        if end:
            clauses.append("txn_date < ?");  params.append(end.strftime(DATE_FORMAT))
        if category:
            clauses.append("category = ?"); params.append(category.strip().lower())
        if ttype:
            clauses.append("ttype = ?"); params.append(ttype.value)
        where = f" WHERE {' AND '.join(clauses)}" if clauses else ""
        query = f"SELECT id, amount, category, ttype, note, txn_date FROM transactions{where} ORDER BY txn_date"
        with self._connect() as conn:
            for row in conn.execute(query, params):
                yield Transaction.from_row(tuple(row))

    def count_transactions(self) -> int:
        with self._connect() as conn:
            (count,) = conn.execute("SELECT COUNT(*) FROM transactions").fetchone()
            return int(count)

    @log_call
    def upsert_budget(self, budget: Budget) -> None:
        with self._connect() as conn:
            conn.execute(
                "INSERT INTO budgets (category, monthly_limit) VALUES (?, ?)"
                " ON CONFLICT(category) DO UPDATE SET monthly_limit = excluded.monthly_limit",
                (budget.category, budget.monthly_limit)
            )

    def all_budgets(self):
        with self._connect() as conn:
            rows = conn.execute("SELECT category, monthly_limit FROM budgets ORDER BY category").fetchall()
        return [Budget(r["category"], r["monthly_limit"]) for r in rows]

print("Database initialized at", DEFAULT_DB_PATH)

# --- Finance Manager ---------------------------------------------
class FinanceManager:
    def __init__(self, db_path=DEFAULT_DB_PATH):
        self.db = Database(db_path)

    @validate_positive("amount")
    @log_call
    def add_transaction(self, *, amount, category, ttype, note="", txn_date=None) -> Transaction:
        # If ttype is a string, parse it; else assume it's a TransactionType
        rtype = ttype if isinstance(ttype, TransactionType) else TransactionType.from_string(ttype)
        txn = Transaction(amount=float(amount), category=category, ttype=rtype,
                          note=note, txn_date=txn_date or date.today())
        self.db.add_transaction(txn)
        return txn

    def list_transactions(self, **filters):
        return list(self.db.iter_transactions(**filters))

    @validate_positive("monthly_limit")
    def set_budget(self, *, category, monthly_limit) -> Budget:
        budget = Budget(category=category, monthly_limit=float(monthly_limit))
        self.db.upsert_budget(budget)
        return budget

    def budget_status(self, year: int, month: int):
        start = date(year, month, 1)
        # Compute first day of next month (handles December correctly)
        end_month = month % 12 + 1
        end_year = year + (month // 12)
        end = date(end_year, end_month, 1)
        spent = defaultdict(float)
        for txn in self.db.iter_transactions(start=start, end=end, ttype=TransactionType.EXPENSE):
            spent[txn.category] += txn.amount
        report = []
        for b in self.db.all_budgets():
            used = spent.get(b.category, 0.0)
            report.append({
                "category": b.category,
                "limit": round(b.monthly_limit, 2),
                "spent": round(used, 2),
                "remaining": round(b.monthly_limit - used, 2),
                "used_pct": round(used / b.monthly_limit * 100, 1) if b.monthly_limit else 0.0,
                "over_budget": used > b.monthly_limit,
            })
        return report

    def summary(self, *, start=None, end=None):
        income = expense = 0.0
        by_cat = defaultdict(float)
        count = 0
        for txn in self.db.iter_transactions(start=start, end=end):
            count += 1
            if txn.ttype == TransactionType.INCOME:
                income += txn.amount
            else:
                expense += txn.amount
                by_cat[txn.category] += txn.amount
        if count == 0:
            raise InsufficientDataError("No transactions in the selected range.")
        net = income - expense
        return {
            "transactions": count,
            "total_income": round(income, 2),
            "total_expense": round(expense, 2),
            "net_balance": round(net, 2),
            "savings_rate_pct": round(net / income * 100, 1) if income else 0.0,
            "expense_by_category": {k: round(v, 2) for k, v in
                                     sorted(by_cat.items(), key=lambda kv: kv[1], reverse=True)},
        }

    def export_csv(self, path) -> int:
        rows = list(self.db.iter_transactions())
        with open(path, "w", newline="", encoding="utf-8") as fh:
            writer = csv.writer(fh)
            writer.writerow(["id", "amount", "category", "ttype", "note", "txn_date"])
            for txn in rows:
                writer.writerow(txn.to_row())
        return len(rows)

    def export_json(self, path) -> int:
        data = [t.to_dict() for t in self.db.iter_transactions()]
        Path(path).write_text(json.dumps(data, indent=2), encoding="utf-8")
        return len(data)

print("FinanceManager class ready")

# --- Reporting Utilities -----------------------------------------
def to_dataframe(manager: FinanceManager) -> pd.DataFrame:
    records = [t.to_dict() for t in manager.db.iter_transactions()]
    if not records:
        raise InsufficientDataError("No transactions to analyze.")
    df = pd.DataFrame.from_records(records)
    df["txn_date"] = pd.to_datetime(df["txn_date"])
    # vectorized signed amount
    df["signed"] = df.apply(lambda r: r["amount"] if r["ttype"] == "income" else -r["amount"], axis=1)
    df["month"] = df["txn_date"].dt.to_period("M").astype(str)
    return df.sort_values("txn_date").reset_index(drop=True)

def plot_spending_by_category(manager: FinanceManager):
    df = to_dataframe(manager)
    expenses = df[df["ttype"] == "expense"].groupby("category")["amount"].sum().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(expenses.values, labels=expenses.index, autopct="%1.1f%%", startangle=140)
    ax.set_title("Spending by Category")
    ax.axis("equal")
    return fig

def plot_monthly_trend(manager: FinanceManager):
    df = to_dataframe(manager)
    pivot = df.pivot_table(index="month", columns="ttype", values="amount", aggfunc="sum", fill_value=0)
    # Ensure both income and expense columns exist
    for col in ("income", "expense"):
        if col not in pivot.columns:
            pivot[col] = 0.0
    pivot["net"] = pivot["income"] - pivot["expense"]
    fig, ax = plt.subplots(figsize=(9, 5))
    pivot[["income", "expense"]].plot(kind="bar", ax=ax, color=["#2e8b57", "#c0392b"])
    ax.plot(range(len(pivot)), pivot["net"].values, marker="o", color="#2c3e50", label="net")
    ax.set_title("Monthly Income vs Expense")
    ax.set_ylabel("Amount")
    ax.legend()
    fig.tight_layout()
    return fig

print("Reporting functions ready")

# --- Demo Data & Tests (only run when script is executed directly) ---
if __name__ == "__main__":
    # Remove any old demo database for a fresh start
    if DEFAULT_DB_PATH.exists():
        DEFAULT_DB_PATH.unlink()

    mgr = FinanceManager()

    # Generate synthetic data: 4 months of 4000 income + random expenses
    random.seed(42)
    categories = {
        "food": (8, 60),
        "rent": (1, 1200),
        "transport": (6, 40),
        "utilities": (2, 120),
        "entertainment": (4, 50),
        "shopping": (3, 90)
    }
    today = date.today()
    # Start roughly 3 months ago on the first day of that month
    start_month = (today.replace(day=1) - timedelta(days=90))
    for m in range(4):
        month_day = (start_month + timedelta(days=30 * m)).replace(day=1)
        mgr.add_transaction(amount=4000, category="Salary", ttype="income",
                            note="Monthly pay", txn_date=month_day)
        for cat, (n, cap) in categories.items():
            for _ in range(random.randint(1, n)):
                day = month_day + timedelta(days=random.randint(0, 27))
                mgr.add_transaction(amount=round(random.uniform(10, cap), 2),
                                    category=cat, ttype="expense", txn_date=day)

    # Set up budgets
    mgr.set_budget(category="Food", monthly_limit=400)
    mgr.set_budget(category="Entertainment", monthly_limit=150)
    mgr.set_budget(category="Shopping", monthly_limit=200)

    # Output summary and reports
    total = mgr.db.count_transactions()
    print(f"Total transactions stored: {total}")

    print("\n--- Overall summary (all data) ---")
    summary = mgr.summary()
    for k, v in summary.items():
        print(f"{k:20}: {v}")

    print("\n--- Budget status (current month) ---")
    status = mgr.budget_status(today.year, today.month)
    if status:
        df_status = pd.DataFrame(status)
        print(df_status.to_string(index=False))
    else:
        print("No budgets set for this month.")

    # Export data
    csv_path = BASE_DIR / "transactions.csv"
    json_path = BASE_DIR / "transactions.json"
    print(f"\nExported {mgr.export_csv(csv_path)} rows to '{csv_path.name}'")
    print(f"Exported {mgr.export_json(json_path)} rows to '{json_path.name}'")

    # Run the built-in tests
    print("\nRunning tests...")
    def fresh_manager(name):
        path = BASE_DIR / name
        if path.exists():
            path.unlink()
        return FinanceManager(path)

    try:
        # 1) Negative amount should fail
        try:
            Transaction(amount=-5, category="food", ttype="expense")
            assert False, "Negative amount should raise ValidationError"
        except ValidationError:
            pass

        # 2) Enum parsing is case-insensitive
        assert TransactionType.from_string("INCOME") == TransactionType.INCOME

        # 3) signed_amount convention
        assert Transaction(amount=10, category="x", ttype="income").signed_amount == 10
        assert Transaction(amount=10, category="x", ttype="expense").signed_amount == -10

        # 4) Round-trip via fresh DB
        tmp_mgr = fresh_manager("test.db")
        tmp_mgr.add_transaction(amount=100, category="salary", ttype="income")
        tmp_mgr.add_transaction(amount=30, category="food", ttype="expense")
        s = tmp_mgr.summary()
        assert s["total_income"] == 100 and s["total_expense"] == 30 and s["net_balance"] == 70

        # 5) Empty DB summary raises error
        empty_mgr = fresh_manager("empty.db")
        try:
            empty_mgr.summary()
            assert False, "Empty summary should raise InsufficientDataError"
        except InsufficientDataError:
            pass

        print("✅ All tests passed!")
    except AssertionError as e:
        print("❌ Test failed:", e)


Created: {'amount': 1200, 'category': 'salary', 'ttype': 'income', 'note': '', 'txn_date': '2026-11-14', 'id': '4df9d6bffe11'} | signed: 1200
Database initialized at /tmp/fintrack_demo/fintrack.db
FinanceManager class ready
Reporting functions ready
Total transactions stored: 65

--- Overall summary (all data) ---
transactions        : 65
total_income        : 16000.0
total_expense       : 3818.66
net_balance         : 12181.34
savings_rate_pct    : 76.1
expense_by_category : {'rent': 1748.47, 'utilities': 516.86, 'transport': 455.44, 'shopping': 447.77, 'food': 378.89, 'entertainment': 271.23}

--- Budget status (current month) ---
     category  limit  spent  remaining  used_pct  over_budget
entertainment  150.0  55.16      94.84      36.8        False
         food  400.0  85.39     314.61      21.3        False
     shopping  200.0 153.97      46.03      77.0        False

Exported 65 rows to 'transactions.csv'
Exported 65 rows to 'transactions.json'

Running tests...
✅ All tests p